# 11 — KKT conditions, curvature, and scaling

**Objectives.** Compute first-order optimality residuals, interpret multipliers, and recognize poor scaling. For equalities $g(x)=0$, the Lagrangian is $L=f+\lambda^Tg$ and KKT stationarity is $\nabla f+J_g^T\lambda=0$. Inequalities add sign and complementarity conditions. These are necessary, not generally sufficient, for a local minimum.


In [1]:
# Make repository-local helpers importable without installing a package.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Autoreload picks up edits to tutorial models before each notebook cell runs.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

### Evaluate first-order KKT residuals

The cell constructs the objective gradient with `ca.gradient` and the
equality-constraint Jacobian with `ca.jacobian`. These derivative functions
are evaluated at an analytic solution and passed to the repository's
solver-independent KKT diagnostic.

For

$f(x)=(x_0-1)^2+(x_1-2)^2,\qquad
g(x)=x_0+x_1-2=0,$

define the Lagrangian

$$
\mathcal L(x,\lambda)=f(x)+\lambda g(x).
$$

First-order optimality requires

$$
\nabla_x\mathcal L
=\nabla f(x)+J_g(x)^\mathsf{T}\lambda=0,
\qquad g(x)=0.
$$

The point $x=[0.5,1.5]^\mathsf{T}$, $\lambda=1$ satisfies these
equations, so its stationarity infinity norm should be near machine zero.


In [ ]:
import casadi as ca

from casadi_tutorial.diagnostics import kkt_residuals

x = ca.SX.sym("x", 2)
objective = (x[0] - 1) ** 2 + (x[1] - 2) ** 2
equality_constraint = x[0] + x[1] - 2

objective_gradient = ca.Function(
    "objective_gradient",
    [x],
    [ca.gradient(objective, x)],
)
constraint_jacobian = ca.Function(
    "constraint_jacobian",
    [x],
    [ca.jacobian(equality_constraint, x)],
)

x_solution = [0.5, 1.5]
residuals = kkt_residuals(
    objective_gradient(x_solution),
    constraint_jacobian(x_solution),
    lambda_g=[1.0],
    constraint=[0.0],
    lower=[0.0],
    upper=[0.0],
)
print(residuals)
assert residuals["stationarity_inf"] < 1e-12

## Scaling

If one decision is near $10^{-6}$ and another near $10^6$, a single solver tolerance has incompatible physical meanings. Introduce dimensionless variables $\hat x=(x-x_{ref})/s_x$, scale residuals by their meaningful magnitudes, and keep objective terms comparable. Scaling changes conditioning, not the physical solution.


### Relate scaling to numerical conditioning

This cell compares an extremely ill-scaled diagonal Hessian with an
identity-scaled Hessian. NumPy computes their matrix condition numbers,
which estimate how strongly relative input errors can be amplified in a
linear solve.

For an invertible matrix $H$, the 2-norm condition number is

$$
\kappa_2(H)=\lVert H\rVert_2\lVert H^{-1}\rVert_2
=\frac{\sigma_{\max}(H)}{\sigma_{\min}(H)}.
$$

Thus

$$
\kappa_2\!\left(\operatorname{diag}(10^{-12},10^{12})\right)=10^{24},
\qquad \kappa_2(I)=1.
$$

Scaling variables and residuals toward comparable magnitudes improves the
linear systems solved inside nonlinear optimization algorithms.


In [3]:
import numpy as np

# The matrix condition number quantifies sensitivity to numerical errors.
poorly_scaled_hessian = np.diag([1e-12, 1e12])
scaled_hessian = np.eye(2)
print(
    "condition numbers:",
    np.linalg.cond(poorly_scaled_hessian),
    np.linalg.cond(scaled_hessian),
)
assert np.linalg.cond(scaled_hessian) == 1.0

condition numbers: 1e+24 1.0


**Common mistake:** increasing `max_iter` cannot repair inconsistent units or a singular constraint Jacobian. **Recap:** inspect primal feasibility, stationarity, complementarity, curvature, and scaling separately.
